# Day 14. Causal Mask / Multi-Head Attention

**오늘의 목표**
- GPT가 왜 미래 Token을 보면 안 되는지, **"커닝 테스트"** 로 직접 확인한다
- `torch.tril` 마스크를 만들고, `register_buffer`로 모듈 안에 넣는 nanoGPT 방식을 이해한다
- 마스크는 **Softmax 전에** 걸어야 하는 이유를 안다
- Multi-Head Attention을 두 가지 방법(Head 여러 개 / 한 번에 계산)으로 구현하고, 결과가 같다는 걸 확인한다
- 3·4주차에서 그대로 쓸 `CausalSelfAttention` 모듈을 완성한다

> 2주차 마지막 날입니다. Day 11~13에서 만든 Q/K/V, Scaled Dot-Product, `Head`를 바탕으로 진행해요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

tokens = ["나는", "오늘", "학교에", "간다"]

## 1. 왜 미래를 보면 안 되나?

GPT는 **다음 Token 맞히기**로 학습합니다. (Day 6에서 본 x / y 한 칸 밀기)

```text
위치      0      1      2       3
x      나는   오늘   학교에   간다
y      오늘   학교에  간다    (다음)
```

위치 0("나는")의 정답은 "오늘"입니다. 그런데 Attention에서 위치 0이 위치 1을 볼 수 있다면?
**정답을 그냥 베껴 쓰면 됩니다.** 학습 Loss는 금방 0이 되지만, 실제로 문장을 생성할 때는 미래가 없으니 아무것도 못 합니다.

그래서 **위치 t는 0 ~ t까지만** 볼 수 있어야 합니다.

### 1-1. 커닝 테스트

"미래를 보는지"를 확인하는 간단한 방법이 있습니다.

> **마지막 Token만 바꿔본다 → 앞쪽 위치의 출력이 바뀌면 미래를 보고 있는 것**

마스크가 없는 Attention과 있는 Attention으로 직접 실험해봅시다.

In [ ]:
B, T, C = 1, 4, 8
head_size = 8

Wq = nn.Linear(C, head_size, bias=False)
Wk = nn.Linear(C, head_size, bias=False)
Wv = nn.Linear(C, head_size, bias=False)

def attention(x, causal):
    q, k, v = Wq(x), Wk(x), Wv(x)
    scores = q @ k.transpose(-2, -1) * head_size ** -0.5      # (B, T, T)
    if causal:
        T = x.shape[1]
        mask = torch.tril(torch.ones(T, T))
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ v                                       # (B, T, head_size)

x = torch.randn(B, T, C)
x_changed = x.clone()
x_changed[:, -1] = torch.randn(C)        # 마지막 Token("간다")만 다른 벡터로 바꿈

with torch.no_grad():
    for causal in [False, True]:
        out1 = attention(x, causal)
        out2 = attention(x_changed, causal)
        name = "마스크 있음" if causal else "마스크 없음"
        print(f"[{name}]")
        for t in range(T):
            same = torch.allclose(out1[0, t], out2[0, t])
            if t == T - 1:
                status = "바뀜 (입력을 바꾼 위치라 당연)"
            else:
                status = "그대로" if same else "바뀜 ← 미래를 봤다!"
            print(f"  위치 {t} ({tokens[t]:3s}) 출력: {status}")
        print()

- **마스크 없음**: "간다"를 바꿨더니 "나는", "오늘", "학교에"의 출력까지 전부 바뀌었습니다. 미래를 보고 있다는 증거예요.
- **마스크 있음**: 앞쪽 3개는 그대로이고, 바꾼 위치 3만 달라졌습니다.

연습 문제 Q2에서 이 테스트를 함수로 만들어봅니다.

## 2. Causal Mask 만들기

### 2-1. `torch.tril` → `masked_fill` → `softmax`

Day 5에서 본 `-inf` 트릭 그대로입니다. `tril`은 **아래 삼각형(lower triangle)** 만 남기는 함수예요.

In [ ]:
T = 4
mask = torch.tril(torch.ones(T, T))
print("mask (1 = 볼 수 있음, 0 = 못 봄):")
print(mask)

torch.manual_seed(1)
scores = torch.randn(T, T)
masked = scores.masked_fill(mask == 0, float("-inf"))
weights = F.softmax(masked, dim=-1)

print()
print("마스크 적용한 점수:")
print(masked.round(decimals=2))
print()
print("Softmax 후 각 행의 합:", r(weights.sum(dim=-1)))

In [ ]:
def show_weights(ax, w, title):
    ax.imshow(w, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)), tokens)
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set_xlabel("참고하는 Token (Key)")
    ax.set_ylabel("지금 Token (Query)")
    ax.set_title(title, fontsize=12)
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            v = w[i, j].item()
            ax.text(j, i, f"{v*100:.0f}%", ha="center", va="center",
                    color="white" if v > 0.5 else "black", fontsize=10)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
show_weights(axes[0], F.softmax(scores, dim=-1), "마스크 없음")
show_weights(axes[1], weights, "Causal Mask 적용")
plt.tight_layout()
plt.show()

오른쪽 그림의 **위쪽 삼각형이 전부 0%** 입니다. "나는"은 자기 자신만, "간다"는 4개 전부를 볼 수 있어요.

### 2-2. 순서가 중요하다: 마스크는 Softmax **전에**

"Softmax를 먼저 하고, 미래 칸을 0으로 지우면 되지 않나?" 라고 생각할 수 있습니다. 해봅시다.

In [ ]:
wrong = F.softmax(scores, dim=-1) * mask        # Softmax 먼저, 그 다음 0으로 지우기
right = F.softmax(scores.masked_fill(mask == 0, float("-inf")), dim=-1)

print("Softmax 후에 지운 경우 각 행의 합:", r(wrong.sum(dim=-1)))
print("Softmax 전에 -inf    각 행의 합:", r(right.sum(dim=-1)))

Softmax 후에 지우면 **각 행의 합이 1이 아니게** 됩니다. 미래 칸에 갔던 확률이 그냥 사라져버린 거예요.
"나는"은 원래 100%를 자기 자신에게 줘야 하는데, 훨씬 적은 비율만 남습니다.

`-inf`를 Softmax **전에** 넣으면, Softmax가 남은 칸들끼리 합이 1이 되도록 알아서 다시 나눠줍니다.

### 2-3. 모듈 안에 넣기: `register_buffer` (nanoGPT 방식)

매번 `torch.tril`을 새로 만드는 대신, 모듈을 만들 때 **최대 길이(`block_size`)짜리 마스크를 한 번만** 만들어 둡니다.
입력이 더 짧으면 `[:T, :T]`로 잘라서 씁니다.

In [ ]:
class Head(nn.Module):
    # Day 13의 Head에 Causal Mask를 추가한 버전

    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                            # (B, T, hs)
        q = self.query(x)                                          # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5        # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)                                          # (B, T, hs)
        return wei @ v                                             # (B, T, hs)

head = Head(n_embd=8, head_size=4, block_size=16)
print("tril buffer shape:", tuple(head.tril.shape))
print("학습되는 파라미터:", [name for name, _ in head.named_parameters()])
print("buffer          :", [name for name, _ in head.named_buffers()])

for T in [3, 10, 16]:
    out = head(torch.randn(2, T, 8))
    print(f"T={T:2d} 입력 → 출력 {tuple(out.shape)}")

> 💡 **왜 `register_buffer`?**
> - `tril`은 학습할 값이 아니라서 `nn.Parameter`로 만들면 안 됩니다 (Optimizer가 건드리면 안 됨)
> - 그렇다고 그냥 `self.tril = ...`로 두면 `model.to("mps")`로 GPU에 옮길 때 같이 안 옮겨집니다
> - buffer로 등록하면 **학습은 안 되지만 모델과 함께 움직이고 저장됩니다**
>
> `block_size`보다 긴 입력은 처리할 수 없습니다. 이게 LLM의 **context length**(최대 입력 길이)의 정체 중 하나예요.

### 2-4. 마스크 덕분에 생기는 보너스: 한 번에 T개를 학습

마스크가 있으면 위치 t의 출력은 0~t만 보고 만들어집니다. 그래서 문장 하나를 넣으면 **T개의 "다음 Token 맞히기" 문제를 한 번에** 풀 수 있어요.

```text
한 번의 forward로:
  "나는"              → 다음은? (정답: 오늘)
  "나는 오늘"          → 다음은? (정답: 학교에)
  "나는 오늘 학교에"    → 다음은? (정답: 간다)
  "나는 오늘 학교에 간다" → 다음은? (정답: ...)
```

마스크가 없었다면 이 4개 문제를 따로따로 4번 돌려야 했을 거예요. GPT 학습이 빠른 이유 중 하나입니다.

## 3. Multi-Head Attention

### 3-1. 왜 Head가 여러 개 필요한가?

Head 하나는 각 행마다 **하나의 확률 분포**만 만듭니다. 그런데 Token 사이에는 여러 종류의 관계가 동시에 있어요.
- "간다"의 **주어**는 누구? → "나는"
- "간다" **바로 앞**에 뭐가 있었지? → "학교에"

Head 하나로 두 관계를 다 표현하려면 확률을 반반 나눠야 해서 둘 다 흐려집니다.
Head를 여러 개 두면 **Head마다 다른 관계**를 맡을 수 있어요. 아래는 이해를 돕기 위해 손으로 만든 예시 패턴입니다.

In [ ]:
T = 4
mask = torch.tril(torch.ones(T, T))

# Head A: "바로 앞 Token"을 주로 본다
prev_scores = torch.full((T, T), -2.0)
for i in range(T):
    prev_scores[i, max(i - 1, 0)] = 3.0

# Head B: "첫 Token(주어)"을 주로 본다
subj_scores = torch.full((T, T), -2.0)
subj_scores[:, 0] = 3.0

def causal_softmax(s):
    return F.softmax(s.masked_fill(mask == 0, float("-inf")), dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
show_weights(axes[0], causal_softmax(prev_scores), "Head A: 바로 앞 Token 보기")
show_weights(axes[1], causal_softmax(subj_scores), "Head B: 주어(첫 Token) 보기")
plt.tight_layout()
plt.show()

실제 모델에서는 이런 역할을 사람이 정해주지 않습니다. 학습하면서 **각 Head가 알아서 다른 패턴을 찾아갑니다.**
(실제로 학습된 GPT-2를 분석하면 "바로 앞 Token 보기" 같은 Head가 정말 발견돼요.)

### 3-2. 구현 A: Head 여러 개 + Concat + Linear

커리큘럼 그림 그대로입니다.

```text
Head 1 ─┐
Head 2 ─┤
Head 3 ─┼→ Concat → Linear(proj)
Head 4 ─┘
```

핵심 규칙: **`head_size = n_embd // n_head`**. 그래야 Concat 결과가 다시 `n_embd`가 됩니다.

In [ ]:
class MultiHeadAttentionA(nn.Module):

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, n_head * head_size) = (B, T, C)
        return self.proj(out)

B, T, C, n_head, block_size = 2, 4, 32, 4, 16
torch.manual_seed(0)
mha_a = MultiHeadAttentionA(C, n_head, block_size)
x = torch.randn(B, T, C)

print("head_size =", C // n_head)
for i, h in enumerate(mha_a.heads):
    print(f"Head {i+1} 출력:", tuple(h(x).shape))
concat = torch.cat([h(x) for h in mha_a.heads], dim=-1)
print("Concat     :", tuple(concat.shape))
print("proj 후    :", tuple(mha_a(x).shape))

> 💡 `proj`(마지막 Linear)는 각 Head가 따로 찾아온 정보를 **섞어서 하나로 정리**하는 역할입니다.

구현 A는 이해하기 쉽지만, Head 개수만큼 반복문을 돌고 작은 행렬 곱을 여러 번 합니다. **느려요.**

### 3-3. 구현 B: 한 번에 계산하기 (nanoGPT 방식) ⚠️ 이번 과정에서 가장 어려운 shape 다루기

아이디어:
1. Q, K, V를 Head별로 따로 만들지 않고 **Linear 하나(`C → 3C`)로 한 번에** 만든다
2. 길이 C 벡터를 **`n_head`개 조각으로 잘라서** Head 축을 새로 만든다
3. `(B, n_head, T, head_size)` 모양으로 바꾸면, `@`가 **모든 Head를 동시에** 계산해준다 (Day 2: `@`는 마지막 두 축으로 곱하고 앞 축은 그대로)

한 줄씩 shape를 찍어가며 따라가봅시다.

In [ ]:
torch.manual_seed(0)
B, T, C, n_head = 2, 5, 32, 4     # T와 n_head를 다르게 둬야 shape 변화가 잘 보여요
hs = C // n_head
x = torch.randn(B, T, C)

c_attn = nn.Linear(C, 3 * C, bias=False)

# ① Q, K, V를 한 번에
qkv = c_attn(x)
print("① c_attn(x)              :", tuple(qkv.shape), "  (B, T, 3C)")

# ② 마지막 축을 3등분
q, k, v = qkv.split(C, dim=2)
print("② split → q              :", tuple(q.shape), "   (B, T, C)")

# ③ C를 (n_head, hs)로 쪼개기
q = q.view(B, T, n_head, hs)
print("③ view                   :", tuple(q.shape), " (B, T, n_head, hs)")

# ④ Head 축을 앞으로: T와 n_head 자리 바꾸기
q = q.transpose(1, 2)
print("④ transpose(1, 2)        :", tuple(q.shape), " (B, n_head, T, hs)")

k = k.view(B, T, n_head, hs).transpose(1, 2)
v = v.view(B, T, n_head, hs).transpose(1, 2)
print("   k, v도 똑같이          :", tuple(k.shape), tuple(v.shape))

**③ `view`에서 무슨 일이?** 길이 32짜리 벡터를 8개씩 4덩어리로 자른 것뿐입니다. 숫자는 하나도 안 바뀌어요.

```text
Token 하나의 q (길이 32):  [ q0 ... q7 | q8 ... q15 | q16 ... q23 | q24 ... q31 ]
                            Head 1       Head 2        Head 3        Head 4
```

**④ `transpose`는 왜?** `@`는 **마지막 두 축**으로 행렬 곱을 합니다. Attention은 `(T, hs) @ (hs, T)`가 되어야 하니까 마지막 두 축이 `(T, hs)`여야 해요.
`(B, T, n_head, hs)`에서는 마지막 두 축이 `(n_head, hs)`라서 안 되고, Head 축을 앞으로 빼야 합니다.

In [ ]:
# ⑤ Attention 점수: 모든 Head를 한 번에
att = q @ k.transpose(-2, -1) * hs ** -0.5
print("⑤ q @ kᵀ                 :", tuple(att.shape), " (B, n_head, T, T)  ← Head마다 (T×T) 표")

# ⑥ Causal Mask + Softmax
tril = torch.tril(torch.ones(T, T))
att = att.masked_fill(tril == 0, float("-inf"))
att = F.softmax(att, dim=-1)
print("⑥ mask + softmax         :", tuple(att.shape))

# ⑦ Value와 곱하기
y = att @ v
print("⑦ att @ v                :", tuple(y.shape), " (B, n_head, T, hs)")

# ⑧ Head 축을 다시 뒤로
y = y.transpose(1, 2)
print("⑧ transpose(1, 2)        :", tuple(y.shape), " (B, T, n_head, hs)")

# ⑨ Head들을 이어붙이기 (= Concat)
y = y.contiguous().view(B, T, C)
print("⑨ contiguous().view      :", tuple(y.shape), "   (B, T, C)")

> 💡 **`contiguous()`는 왜 필요한가?**
> `transpose`는 데이터를 실제로 옮기지 않고 "읽는 순서"만 바꿉니다. `view`는 데이터가 메모리에 순서대로 놓여 있어야 동작해요.
> `contiguous()`가 데이터를 새 순서대로 다시 복사해줍니다. 빼먹으면 아래처럼 에러가 나요.

In [ ]:
try:
    att_tmp = torch.randn(B, n_head, T, hs).transpose(1, 2)
    att_tmp.view(B, T, C)
except RuntimeError as e:
    print("RuntimeError:", str(e)[:90], "...")

**전체 흐름 한눈에 보기**

```text
x                (B, T, C)
 ↓ c_attn        (B, T, 3C)
 ↓ split         q, k, v 각각 (B, T, C)
 ↓ view          (B, T, n_head, hs)
 ↓ transpose     (B, n_head, T, hs)
 ↓ q @ kᵀ        (B, n_head, T, T)
 ↓ mask, softmax (B, n_head, T, T)
 ↓ @ v           (B, n_head, T, hs)
 ↓ transpose     (B, T, n_head, hs)
 ↓ view          (B, T, C)          ← Concat과 같음
 ↓ proj          (B, T, C)
```

### 3-4. 정말 구현 A와 같은가?

구현 A의 Weight를 구현 B로 그대로 복사해서 결과를 비교해봅시다.
`c_attn.weight`는 `(3C, C)`이고, 위에서부터 **[Q 전체 | K 전체 | V 전체]** 순서이며, 각 덩어리 안에서는 **Head 1, 2, 3, 4** 순서입니다.

In [ ]:
c_attn = nn.Linear(C, 3 * C, bias=False)
proj_b = nn.Linear(C, C)

with torch.no_grad():
    Wq_all = torch.cat([h.query.weight for h in mha_a.heads], dim=0)   # (C, C)
    Wk_all = torch.cat([h.key.weight   for h in mha_a.heads], dim=0)
    Wv_all = torch.cat([h.value.weight for h in mha_a.heads], dim=0)
    c_attn.weight.copy_(torch.cat([Wq_all, Wk_all, Wv_all], dim=0))   # (3C, C)
    proj_b.weight.copy_(mha_a.proj.weight)
    proj_b.bias.copy_(mha_a.proj.bias)

def mha_b_forward(x):
    B, T, C = x.shape
    q, k, v = c_attn(x).split(C, dim=2)
    q = q.view(B, T, n_head, hs).transpose(1, 2)
    k = k.view(B, T, n_head, hs).transpose(1, 2)
    v = v.view(B, T, n_head, hs).transpose(1, 2)
    att = (q @ k.transpose(-2, -1)) * hs ** -0.5
    att = att.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))
    att = F.softmax(att, dim=-1)
    y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
    return proj_b(y)

x = torch.randn(B, T, C)
with torch.no_grad():
    print("구현 A == 구현 B ?", torch.allclose(mha_a(x), mha_b_forward(x), atol=1e-6))

### 3-5. 파라미터 개수: Head를 늘리면 비싸질까?

In [ ]:
def count(m):
    return sum(p.numel() for p in m.parameters())

single = Head(C, C, block_size)                # head_size = C 인 큰 Head 하나
four   = MultiHeadAttentionA(C, 4, block_size)
eight  = MultiHeadAttentionA(C, 8, block_size)

print(f"Head 1개 (hs={C})  Q/K/V: {count(single):,}")
print(f"Head 4개 (hs={C//4})   Q/K/V: {count(four) - count(four.proj):,}   + proj {count(four.proj):,}")
print(f"Head 8개 (hs={C//8})   Q/K/V: {count(eight) - count(eight.proj):,}   + proj {count(eight.proj):,}")

Q/K/V 파라미터는 **Head 개수와 상관없이 항상 `3 × C × C`** 입니다. 큰 Head 하나를 작은 Head 여러 개로 **나눠 쓰는 것**일 뿐이에요.
그래서 Multi-Head는 거의 공짜로 "여러 관계를 동시에 보는 능력"을 얻습니다.

## 4. 완성본: `CausalSelfAttention`

3주차 Transformer Block과 4주차 mini GPT에서 **이 클래스를 그대로** 가져다 씁니다.
(4주차에서는 여기에 Dropout만 추가할 거예요.)

In [ ]:
class CausalSelfAttention(nn.Module):

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd는 n_head로 나누어떨어져야 합니다"
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)   # Q, K, V 한 번에
        self.proj   = nn.Linear(n_embd, n_embd)                   # Head 합친 뒤 정리
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        hs = C // self.n_head

        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)        # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * hs ** -0.5             # (B, nh, T, T)
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v                                              # (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)         # (B, T, C)
        return self.proj(y)

attn = CausalSelfAttention(n_embd=32, n_head=4, block_size=16)
out = attn(torch.randn(2, 10, 32))
print("출력:", tuple(out.shape))
print("파라미터:", count(attn))

> 💡 PyTorch 2.0부터는 ⑤~⑦을 한 줄로 해주는 `F.scaled_dot_product_attention(q, k, v, is_causal=True)`가 있습니다.
> 결과는 같고 훨씬 빠릅니다(Flash Attention). 지금은 원리를 보려고 직접 썼지만, 실제 nanoGPT도 이 함수를 씁니다.

In [ ]:
torch.manual_seed(0)
q = torch.randn(2, 4, 6, 8); k = torch.randn(2, 4, 6, 8); v = torch.randn(2, 4, 6, 8)

att = (q @ k.transpose(-2, -1)) * 8 ** -0.5
att = att.masked_fill(torch.tril(torch.ones(6, 6)) == 0, float("-inf"))
manual = F.softmax(att, dim=-1) @ v

fast = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("직접 구현 == F.scaled_dot_product_attention ?", torch.allclose(manual, fast, atol=1e-5))

## 5. LLM과의 연결: 2주차 전체 흐름 정리

Day 8부터 오늘까지 배운 것을 한 번에 이어봅시다.

```text
문장 → Tokenizer → Token ID → Embedding → + Position → Self-Attention
       (Day 8)      (Day 8)    (Day 9)      (Day 10)     (Day 11~14)
```

In [ ]:
torch.manual_seed(0)
text = "나는 오늘 학교에 간다"

# Day 8: Tokenizer
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
ids = torch.tensor([[stoi[ch] for ch in text]])          # (B=1, T)
B, T = ids.shape

n_embd, n_head, block_size = 32, 4, 16
vocab_size = len(chars)

# Day 9: Token Embedding / Day 10: Position Embedding
tok_emb = nn.Embedding(vocab_size, n_embd)
pos_emb = nn.Embedding(block_size, n_embd)

# Day 11~14: Causal Multi-Head Self-Attention
attn = CausalSelfAttention(n_embd, n_head, block_size)

x_tok = tok_emb(ids)
x_pos = pos_emb(torch.arange(T))
x = x_tok + x_pos
out = attn(x)

print(f"문장           : '{text}'")
print(f"Token ID       : {ids.shape} → {ids[0].tolist()}")
print(f"Token Embedding: {tuple(x_tok.shape)}  (B, T, C)")
print(f"Position Emb   : {tuple(x_pos.shape)}     (T, C) → broadcast 덧셈")
print(f"x = tok + pos  : {tuple(x.shape)}")
print(f"Self-Attention : {tuple(out.shape)}  ← shape는 그대로, 각 Token이 앞쪽 Token 정보를 담게 됨")

Self-Attention은 입력과 출력의 shape가 `(B, T, C)`로 **똑같습니다.**
그래서 3주차에서 이걸 FFN, Residual, LayerNorm과 묶어 **Transformer Block**을 만들고, 그 Block을 **N번 쌓을** 수 있게 됩니다.

```text
2주차 끝:  Embedding → Self-Attention
3주차:    Embedding → [Self-Attention → FFN] × N → Linear → Softmax → Next Token
```

## 6. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 길이 5인 문장의 Attention 점수에 Causal Mask를 적용하고 Softmax 하세요.
- 위쪽 삼각형(미래)은 전부 0, 각 행의 합은 1이 되어야 합니다

In [ ]:
torch.manual_seed(42)
scores = torch.randn(5, 5)

mask = None      # TODO: (5, 5) 아래 삼각형 마스크
weights = None   # TODO: mask를 적용하고 softmax

assert weights is not None and tuple(weights.shape) == (5, 5), "weights의 shape를 확인하세요"
assert torch.all(torch.triu(weights, diagonal=1) == 0), "미래 칸에 확률이 남아 있어요"
assert torch.allclose(weights.sum(dim=-1), torch.ones(5)), "각 행의 합이 1이 아니에요. 마스크를 거는 순서를 확인하세요"
print("Q1 정답!")

**Q2.** 1-1의 커닝 테스트를 함수로 만드세요.
`sees_future(attn_fn)`은 Attention 함수를 받아서, **미래 Token을 보면 `True`, 안 보면 `False`** 를 돌려줘야 합니다.

힌트: 랜덤 입력 `x`를 만들고, 마지막 Token만 바꾼 `x2`를 만든 뒤, 마지막을 **제외한** 위치들의 출력을 비교하세요.

In [ ]:
def sees_future(attn_fn, T=6, C=32):
    x = torch.randn(1, T, C)
    # TODO
    return None

torch.manual_seed(0)
_m = MultiHeadAttentionA(32, 4, 16)
_c = CausalSelfAttention(32, 4, 16)
_wq, _wk, _wv = nn.Linear(32, 32, bias=False), nn.Linear(32, 32, bias=False), nn.Linear(32, 32, bias=False)
def attention_no_mask(x):
    w = F.softmax(_wq(x) @ _wk(x).transpose(-2, -1) / 32 ** 0.5, dim=-1)
    return w @ _wv(x)

with torch.no_grad():
    r1, r2, r3 = sees_future(attention_no_mask), sees_future(_m), sees_future(_c)
assert r1 is True, "마스크 없는 Attention을 제대로 잡아내지 못했어요"
assert r2 is False and r3 is False, "마스크가 있는 Attention을 미래를 본다고 판단했어요"
print("Q2 정답!")

**Q3.** `n_embd = 64`, `n_head = 8`, 입력 `(B=2, T=10)`인 `CausalSelfAttention`에서 shape를 예측하세요. (실행 전에!)

In [ ]:
head_size       = "?"   # TODO: 숫자 하나
q_after_split   = "?"   # TODO: view + transpose 후 q의 shape (튜플)
att_shape       = "?"   # TODO: q @ kᵀ 의 shape (튜플)
c_attn_weight   = "?"   # TODO: c_attn.weight 의 shape (튜플)  (힌트: nn.Linear는 (out, in))

_B, _T, _C, _nh = 2, 10, 64, 8
_hs = _C // _nh
_x = torch.randn(_B, _T, _C)
_q = _x.view(_B, _T, _nh, _hs).transpose(1, 2)
assert head_size == _hs, "head_size를 다시 계산해보세요"
assert q_after_split == tuple(_q.shape), "q의 shape를 다시 생각해보세요 (3-3의 ④)"
assert att_shape == tuple((_q @ _q.transpose(-2, -1)).shape), "Attention 점수의 shape를 다시 생각해보세요 (3-3의 ⑤)"
assert c_attn_weight == tuple(CausalSelfAttention(_C, _nh, 16).c_attn.weight.shape), "c_attn.weight shape를 다시 생각해보세요"
print("Q3 정답!")

**Q4.** Head 나누기 / 다시 합치기를 직접 해보세요.
- `split_heads`: `(B, T, C)` → `(B, n_head, T, hs)`
- `merge_heads`: `(B, n_head, T, hs)` → `(B, T, C)`
- 나눴다가 다시 합치면 **원래 텐서와 완전히 같아야** 합니다

In [ ]:
def split_heads(x, n_head):
    B, T, C = x.shape
    return None  # TODO

def merge_heads(x):
    B, nh, T, hs = x.shape
    return None  # TODO

x = torch.randn(2, 6, 32)
s = split_heads(x, n_head=4)
assert s is not None and tuple(s.shape) == (2, 4, 6, 8), "split_heads 결과 shape가 달라요"
assert torch.equal(s[:, 1], x[:, :, 8:16]), "Head 1이 x의 올바른 조각을 담고 있지 않아요"
m = merge_heads(s)
assert m is not None and torch.equal(m, x), "다시 합친 결과가 원래 x와 달라요"
print("Q4 정답!")

## 7. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Causal Mask는 왜 필요한가? 없으면 학습 중에 무슨 일이 생기는가?
2. 마스크를 Softmax **후에** 0으로 지우면 어떤 문제가 생기는가?
3. `tril`을 `nn.Parameter`가 아니라 `register_buffer`로 등록하는 이유는?
4. Multi-Head Attention은 왜 필요한가? Head를 늘리면 파라미터도 늘어나는가?
5. `(B, T, C)`가 `(B, n_head, T, hs)`가 되기까지 어떤 연산을 거치는가? `transpose`는 왜 하는가?
6. (2주차 정리) "나는 오늘 학교에 간다"가 Self-Attention 출력이 되기까지의 과정을 shape와 함께 설명해보세요.

**내 답:**

1.
2.
3.
4.
5.
6.